In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
from pathlib import Path

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True

Required setup and data

In [3]:
np.random.seed(0)

c = 3e8
fc = 3.5e9
d0 = 1.0

FSPL_d0_theory = 20 * np.log10(4 * np.pi * d0 * fc / c)


def synthesize_pathloss(
    n_true, sigma_true, n_samples=150,
    d_min=10.0, d_max=1800.0, seed=1
):
    """Generate synthetic path-loss measurements using the CI model."""
    rng = np.random.default_rng(seed)

    log_d = rng.uniform(
        np.log10(d_min), np.log10(d_max), size=n_samples
    )
    d = 10 ** log_d

    shadowing = rng.normal(0, sigma_true, size=n_samples)

    PL = (
        FSPL_d0_theory
        + 10 * n_true * np.log10(d / d0)
        + shadowing
    )

    return d, PL


DATA_PATH = Path("pathloss.txt")

# Used only when pathloss.txt is not available.
N_TRUE = 3.0
SIGMA_TRUE = 6.0

if DATA_PATH.exists():
    data = np.loadtxt(DATA_PATH)
    d_meas = data[:, 0]
    PL_meas = data[:, 1]
else:
    d_meas, PL_meas = synthesize_pathloss(
        N_TRUE, SIGMA_TRUE
    )

MyLinearRegression

In [4]:
class MyLinearRegression:
    """Simple linear regression implemented using NumPy."""

    def __init__(self):
        self.theta = None
        self.loss_history_ = None

    @staticmethod
    def _design_matrix(x):
        x = np.asarray(x).reshape(-1)
        return np.column_stack([np.ones_like(x), x])

    @staticmethod
    def _mse(Phi, y, theta):
        residual = Phi @ theta - y
        return np.mean(residual ** 2)

    def fit(
        self, x, y, method="closed_form",
        lr=0.05, epochs=1500, random_state=0
    ):
        Phi = self._design_matrix(x)
        y = np.asarray(y).reshape(-1)
        n = len(y)

        if method == "closed_form":
            self.theta = np.linalg.solve(
                Phi.T @ Phi, Phi.T @ y
            )
            self.loss_history_ = None

        elif method == "batch_gd":
            theta = np.zeros(2)
            history = np.empty(epochs)

            for epoch in range(epochs):
                residual = Phi @ theta - y
                grad = (2.0 / n) * (Phi.T @ residual)
                theta -= lr * grad
                history[epoch] = self._mse(
                    Phi, y, theta
                )

            self.theta = theta
            self.loss_history_ = history

        elif method == "sgd":
            rng = np.random.default_rng(random_state)
            theta = np.zeros(2)
            history = np.empty(epochs)

            for epoch in range(epochs):
                order = rng.permutation(n)

                for i in order:
                    phi_i = Phi[i]
                    residual_i = phi_i @ theta - y[i]
                    grad_i = 2.0 * residual_i * phi_i
                    theta -= lr * grad_i

                history[epoch] = self._mse(
                    Phi, y, theta
                )

            self.theta = theta
            self.loss_history_ = history

        else:
            raise ValueError(
                "method must be 'closed_form', 'batch_gd', or 'sgd'"
            )

        return self

    def predict(self, x):
        Phi = self._design_matrix(x)
        return Phi @ self.theta

    @property
    def intercept_(self):
        return self.theta[0]

    @property
    def slope_(self):
        return self.theta[1]


fit the three models

In [5]:
x_feat = np.log10(d_meas / d0)
y_target = PL_meas

LR = 0.05
EPOCHS = 1500

model_cf = MyLinearRegression().fit(
    x_feat, y_target, method="closed_form"
)

model_gd = MyLinearRegression().fit(
    x_feat, y_target,
    method="batch_gd",
    lr=LR,
    epochs=EPOCHS
)

model_sgd = MyLinearRegression().fit(
    x_feat, y_target,
    method="sgd",
    lr=LR,
    epochs=EPOCHS,
    random_state=0
)

Estimate Shadowing Variance

In [6]:
def estimate_sigma(model, x, y):
    residuals = y - model.predict(x)
    sigma2_hat = np.mean(residuals ** 2)
    return np.sqrt(sigma2_hat), residuals


sigma_cf, resid_cf = estimate_sigma(model_cf, x_feat, y_target)
sigma_gd, resid_gd = estimate_sigma(model_gd, x_feat, y_target)
sigma_sgd, resid_sgd = estimate_sigma(model_sgd, x_feat, y_target)

summary_table = pd.DataFrame({
    "Method": ["Closed-form", "Batch GD", "SGD"],
    "FSPL(d0) [dB]": [model_cf.intercept_, model_gd.intercept_, model_sgd.intercept_],
    "n": [model_cf.slope_ / 10, model_gd.slope_ / 10, model_sgd.slope_ / 10],
    "sigma [dB]": [sigma_cf, sigma_gd, sigma_sgd],
})
print(summary_table.to_string(index=False))
print(f"\n(Ground truth used to synthesize the data, hidden from the fit: "
      f"FSPL(d0)={FSPL_d0_theory:.3f} dB, n={N_TRUE}, sigma={SIGMA_TRUE} dB)")

     Method  FSPL(d0) [dB]        n  sigma [dB]
Closed-form      42.569518 2.998803    5.800238
   Batch GD      42.568286 2.998856    5.800238
        SGD      41.374047 2.988131    5.972961

(Ground truth used to synthesize the data, hidden from the fit: FSPL(d0)=43.323 dB, n=3.0, sigma=6.0 dB)


Results

In [7]:
summary_table = pd.DataFrame({
    "Method": ["Closed-form", "Batch GD", "SGD"],
    "FSPL(d0) [dB]": [
        model_cf.intercept_,
        model_gd.intercept_,
        model_sgd.intercept_
    ],
    "n": [
        model_cf.slope_ / 10,
        model_gd.slope_ / 10,
        model_sgd.slope_ / 10
    ],
    "sigma [dB]": [
        sigma_cf,
        sigma_gd,
        sigma_sgd
    ],
})

print(summary_table.to_string(index=False))

if not DATA_PATH.exists():
    print(
        f"\nGround truth used only for synthetic data: "
        f"FSPL(d0)={FSPL_d0_theory:.3f} dB, "
        f"n={N_TRUE}, sigma={SIGMA_TRUE} dB"
    )


     Method  FSPL(d0) [dB]        n  sigma [dB]
Closed-form      42.569518 2.998803    5.800238
   Batch GD      42.568286 2.998856    5.800238
        SGD      41.374047 2.988131    5.972961

Ground truth used only for synthetic data: FSPL(d0)=43.323 dB, n=3.0, sigma=6.0 dB
